# Stage 8 — Verify (two-tier)

Rebuild the graph **from the edge list** (not from any saved graph object) and recompute every
headline number, comparing against `data/expected_values.json` written by Stage 6.

**Two tiers (D7):**
- **Exact tier** — deterministic results (persistent-predicate counts, first-after-recall, max gap,
  sensitivity subsets, DAG, depth). Must match to the integer.
- **Drift-tolerant tier** — quantities that move on a live re-fetch (corpus size, edge count,
  component sizes). Pass if within tolerance; both numbers printed so drift is visible.

Set `DATA_DIR` below. Network: none.

**Note (2026-09):** this notebook recomputes the headline numbers from the SAME run's
`data/*.csv` and compares them to the SAME run's `expected_values.json`. It is a self-consistency
check, not independent reproduction — the reproduction result is `python compare_to_reference.py`
run by a second analyst on the frozen inputs.

**2026-10 (D8):** the post-recall rule is now "citing 510(k) received after the predicate's recall
was public". This notebook re-derives both dates from the frozen snapshot with its own code (a
vectorised merge, not Stage 6's loop) and also re-checks the manuscript-v5 year rule
(`sens_year_rule`).

**2026-10-27:** Tier 3 re-checks the headline numbers of the post hoc notebooks `06b_denominators` and `06c_recall_timing` (run them before this notebook).

**2026-11-18:** Tier 4 re-checks the design-related recall harm analysis (`06e_design_harm`, registered addendum v1.0). Run 06e before this notebook.

In [1]:
DATA_DIR = "./data"

In [2]:
import os, json
import pandas as pd
import networkx as nx
import gzip

E = json.load(open(os.path.join(DATA_DIR, "expected_values.json")))
corpus = pd.read_csv(os.path.join(DATA_DIR, "corpus.csv"), dtype=str)
corpus["decision_year"] = pd.to_numeric(corpus["decision_year"], errors="coerce")
edges = pd.read_csv(os.path.join(DATA_DIR, "predicate_edges.csv"), dtype=str)
edges_rule1 = pd.read_csv(os.path.join(DATA_DIR, "predicate_edges_rule1.csv"), dtype=str)
recalled = pd.read_csv(os.path.join(DATA_DIR, "recalled_nodes.csv"), dtype=str)

year = dict(zip(corpus["k_number"], corpus["decision_year"]))
recall_date = dict(zip(recalled["k_number"], pd.to_datetime(recalled["event_date_initiated"], errors="coerce")))
recall_class = dict(zip(recalled["k_number"], recalled["worst_class"]))
recalled_set = set(recalled["k_number"]) & set(corpus["k_number"])

results = []
def check(name, expected, computed, tier="exact", tol=0.0):
    if tier == "exact":
        ok = (expected == computed)
    else:
        try:
            ok = abs(expected - computed) <= tol * max(1, abs(expected))
        except TypeError:
            ok = (expected == computed)
    tag = "PASS" if ok else ("PASS(drift)" if tier == "drift" and
           abs(expected-computed) <= 0.05*max(1,abs(expected)) else "FAIL")
    if tier == "drift" and not ok:
        tag = "FAIL"
    results.append(ok or (tier == "drift" and abs(expected-computed) <= 0.02*max(1,abs(expected))))
    flag = tag if (ok or tag.startswith("PASS")) else "FAIL"
    print(f"  [{flag:11}] {name:38} expected={expected!s:16} computed={computed}")

# ---- 2026-10 (D8): dates for the post-recall rule, re-derived independently of Stage 6 ----
SNAP = os.path.join(os.path.dirname(os.path.abspath(DATA_DIR)), "snapshot")
k510 = pd.DataFrame(json.load(gzip.open(os.path.join(SNAP, "510k_raw.json.gz"), "rt")))
received = dict(zip(k510["k_number"], pd.to_datetime(k510["date_received"], errors="coerce")))
rr = pd.DataFrame(json.load(gzip.open(os.path.join(SNAP, "recall_raw.json.gz"), "rt")))
en = pd.DataFrame(json.load(gzip.open(os.path.join(SNAP, "enforcement_raw.json.gz"), "rt")))
en = en.drop_duplicates("recall_number", keep="last")[["recall_number", "report_date"]]
rr = rr.merge(en, left_on="product_res_number", right_on="recall_number", how="left")
posted = rr["event_date_posted"] if "event_date_posted" in rr else pd.Series(pd.NaT, index=rr.index)
rr["public"] = pd.concat([pd.to_datetime(posted, errors="coerce"),
                          pd.to_datetime(rr["report_date"], errors="coerce")], axis=1).min(axis=1)
rr = rr.explode("k_numbers").dropna(subset=["k_numbers", "public"])
rr["k"] = rr["k_numbers"].astype(str).str.strip().str.upper()
recall_public = rr[rr["k"].isin(recalled_set)].groupby("k")["public"].min().to_dict()

def post(A, B, rule):
    if rule == "year":
        rd, by = recall_date.get(A), year.get(B)
        return pd.notna(rd) and pd.notna(by) and by > rd.year
    pub, rec = recall_public.get(A, pd.NaT), received.get(B, pd.NaT)
    return pd.notna(pub) and pd.notna(rec) and pub < rec

### Rebuild graph and recompute

In [3]:
G = nx.DiGraph(); G.add_nodes_from(corpus["k_number"])
for _, r in edges.iterrows():
    if r["predicate_knumber"] in G and r["device_knumber"] in G:
        G.add_edge(r["predicate_knumber"], r["device_knumber"], confidence=r["confidence"])

def analysis(edge_df, recalled_ok, rule=E.get("post_recall_rule", "year")):
    H = nx.DiGraph(); H.add_nodes_from(corpus["k_number"])
    for _, r in edge_df.iterrows():
        if r["predicate_knumber"] in H and r["device_knumber"] in H:
            H.add_edge(r["predicate_knumber"], r["device_knumber"])
    preds, cits, downstream = set(), 0, set()
    for A in recalled_ok:
        if A not in H: continue
        rd = recall_date.get(A)
        if pd.isna(rd): continue
        for B in H.successors(A):
            if B in recalled_set: continue
            if post(A, B, rule):
                preds.add(A); cits += 1; downstream.add(B)
    return [len(preds), cits, len(downstream)]

recalled_nodes = recalled_set & set(G.nodes())
connected = [n for n in G if G.degree(n) > 0]
largest_wcc = max((len(c) for c in nx.weakly_connected_components(G)), default=0)
depth = nx.dag_longest_path_length(nx.condensation(G))  # match Stage 6: depth on condensation (graph is a near-DAG)
base = analysis(edges, recalled_nodes)
cii = {A for A in recalled_nodes if recall_class.get(A) in ("Class I", "Class II")}
hc = edges[edges["confidence"] == "NEAR_CUE"]   # was SECTION_HEADED

RULE = E.get("post_recall_rule", "year")
first_after = 0
for A in recalled_nodes:
    if A not in G or pd.isna(recall_date.get(A)): continue
    dated = year if RULE == "year" else received
    cites = [B for B in G.successors(A) if pd.notna(dated.get(B, pd.NaT))]
    if cites and all(post(A, B, RULE) for B in cites): first_after += 1
max_gap = 0.0
for A in recalled_nodes:
    rd = recall_date.get(A)
    if A not in G or pd.isna(rd): continue
    cy = [year.get(B) for B in G.successors(A) if pd.notna(year.get(B))]
    if cy: max_gap = max(max_gap, max(cy) - rd.year)

### Tier 1 — deterministic (exact)

In [4]:
print("=== deterministic tier (must match exactly) ===")
check("is a DAG", E["is_dag"], nx.is_directed_acyclic_graph(G))
check("max chain depth", E["max_chain_depth"], depth)
check("recalled nodes", E["recalled_nodes"], len(recalled_nodes))
check("persistent predicates", E["persistent_predicates"], base[0])
check("post-recall citations", E["post_recall_citations"], base[1])
check("downstream devices", E["downstream_devices"], base[2])
check("first cited after recall", E["first_after_recall"], first_after)
check("max recall-to-latest (yr)", E["max_gap_years"], round(max_gap, 1))
check("sensitivity class I/II", tuple(E["sens_class_I_II"]), tuple(analysis(edges, cii)))
check("sensitivity high-conf", tuple(E["sens_high_conf"]), tuple(analysis(hc, recalled_nodes)))
check("sensitivity combined", tuple(E["sens_combined"]), tuple(analysis(hc, cii)))
check("sensitivity maximal rule (R1)", tuple(E["sens_maximal_rule"]), tuple(analysis(edges_rule1, recalled_nodes)))
check("edges (Rule 1)", E["edges_rule1"], len(edges_rule1.drop_duplicates(subset=["predicate_knumber","device_knumber"])))
if "sens_year_rule" in E:
    check("sensitivity year rule (v5)", tuple(E["sens_year_rule"]), tuple(analysis(edges, recalled_nodes, rule="year")))
    check("recalled with public date", E["recalled_with_public_date"], len(recall_public))

=== deterministic tier (must match exactly) ===
  [PASS       ] is a DAG                               expected=False            computed=False
  [PASS       ] max chain depth                        expected=30               computed=30
  [PASS       ] recalled nodes                         expected=1691             computed=1691
  [PASS       ] persistent predicates                  expected=893              computed=893
  [PASS       ] post-recall citations                  expected=2553             computed=2553
  [PASS       ] downstream devices                     expected=1709             computed=1709
  [PASS       ] first cited after recall               expected=356              computed=356
  [PASS       ] max recall-to-latest (yr)              expected=21               computed=21
  [PASS       ] sensitivity class I/II                 expected=(680, 1883, 1298) computed=(680, 1883, 1298)
  [PASS       ] sensitivity high-conf                  expected=(789, 1949, 1418) comput

### Tier 2 — drift-tolerant (live-refetch counts; ±2% ok)

In [5]:
print("\n=== formerly drift-tolerant; exact since the text corpus was frozen (D5) ===")
check("corpus devices", E["corpus_devices"], len(corpus))
check("edges", E["edges"], G.number_of_edges())
check("connected nodes", E["connected_nodes"], len(connected))
check("largest component", E["largest_component"], largest_wcc)


=== formerly drift-tolerant; exact since the text corpus was frozen (D5) ===
  [PASS       ] corpus devices                         expected=15581            computed=15581
  [PASS       ] edges                                  expected=28562            computed=28562
  [PASS       ] connected nodes                        expected=11923            computed=11923
  [PASS       ] largest component                      expected=11588            computed=11588


### Tier 3 — post hoc notebooks 06b and 06c (recomputed independently)
Re-derives the headline numbers of `06b_denominators` and `06c_recall_timing` with separate code and
compares them to `data/denominators_summary.json` and `data/timing_summary.json`. Run 06b and 06c
before this notebook.

In [6]:
import numpy as np
print("\n=== post hoc notebooks 06b / 06c (must match exactly) ===")
for f in ["denominators_summary.json", "timing_summary.json"]:
    if not os.path.exists(os.path.join(DATA_DIR, f)):
        check(f"{f} present (run 06b and 06c first)", True, False)
if all(os.path.exists(os.path.join(DATA_DIR, f)) for f in ["denominators_summary.json", "timing_summary.json"]):
    DS = json.load(open(os.path.join(DATA_DIR, "denominators_summary.json")))
    TS = json.load(open(os.path.join(DATA_DIR, "timing_summary.json")))
    decided = pd.to_datetime(corpus["decision_date"], errors="coerce")
    END = decided.max()
    # --- 06b: evaluable devices, persistence and follow-up, from this notebook's own dates ---
    ev = sorted(A for A in recalled_set if pd.notna(recall_date.get(A)) and pd.notna(recall_public.get(A, pd.NaT)))
    def persists(A):
        P = recall_public[A]
        return any(B not in recalled_set and pd.notna(received.get(B, pd.NaT)) and received[B] > P for B in G.successors(A))
    def had_prior(A):
        P = recall_public[A]
        return any(pd.notna(received.get(B, pd.NaT)) and received[B] < P for B in G.successors(A))
    per = pd.DataFrame({"k": ev, "persist": [persists(A) for A in ev], "prior": [had_prior(A) for A in ev],
                        "fu": [(END - recall_public[A]).days / 365.25 for A in ev]})
    pct = lambda s: round(100 * s["persist"].mean(), 1)
    check("06b evaluable devices", DS["evaluable"], len(per))
    check("06b persistent", DS["persistent"], int(per["persist"].sum()))
    check("06b % persistent, all", DS["pct_all"], pct(per))
    check("06b % persistent, >=5 y follow-up", DS["pct_fu5"], pct(per[per["fu"] >= 5]))
    check("06b % persistent, prior use", DS["pct_prior"], pct(per[per["prior"]]))
    check("06b % persistent, no prior use", DS["pct_no_prior"], pct(per[~per["prior"]]))
    # --- 06c: timing categories, vectorised ---
    decision = dict(zip(k510["k_number"], pd.to_datetime(k510["decision_date"], errors="coerce")))
    t = edges.drop_duplicates(["predicate_knumber", "device_knumber"]).rename(columns={"predicate_knumber": "a", "device_knumber": "b"})
    t = t[t["a"].isin(recalled_set) & ~t["b"].isin(recalled_set) & t["b"].isin(set(corpus["k_number"]))].copy()
    t["init"], t["pub"] = t["a"].map(recall_date), t["a"].map(recall_public)
    t["rec"], t["dec"] = t["b"].map(received), t["b"].map(decision)
    t = t[t["init"].notna() & t["dec"].notna() & (t["dec"] > t["init"])]
    def cats(lag):
        miss = t["rec"].isna()
        isA = ~miss & t["pub"].notna() & ((t["pub"] + pd.Timedelta(days=lag)) < t["rec"])
        isC = ~miss & ~isA & (t["rec"] < t["init"])
        isU = miss | (~isA & ~isC & ((t["rec"] == t["init"]) | t["pub"].isna()))
        return pd.Series(np.select([isA, isC, isU], ["A", "C", "U"], "B"), index=t.index)
    c0, c180 = cats(0), cats(180)
    for k in ["A", "B", "C", "U"]:
        check(f"06c category {k} citations", TS["primary"][k][0], int((c0 == k).sum()))
    a180 = t[c180 == "A"]
    check("06c lag-180 A triple", tuple(TS["triple_A_lag180"]),
          (a180["a"].nunique(), len(a180), a180["b"].nunique()))
    # same corporate family at submission (firm_families.py map; recalling firms read fresh from the snapshot)
    if "same_family_citations" in TS:
        import firm_families as ff
        app = dict(zip(corpus["k_number"], corpus["applicant"]))
        rf = pd.DataFrame(json.load(gzip.open(os.path.join(SNAP, "recall_raw.json.gz"), "rt"))).explode("k_numbers")
        rf = rf.dropna(subset=["k_numbers"]); rf["k"] = rf["k_numbers"].astype(str).str.strip().str.upper()
        rfirms = rf[rf["k"].isin(recalled_set)].groupby("k")["recalling_firm"].apply(
            lambda s: sorted({x for x in s if isinstance(x, str) and x})).to_dict()
        aA = t[c0 == "A"]
        n_fam = sum(any(ff.same_family(n, app.get(b), r) for n in [app.get(a)] + rfirms.get(a, []))
                    for a, b, r in zip(aA["a"], aA["b"], aA["rec"]))
        check("06c same-family A citations", TS["same_family_citations"]["A"], n_fam)


=== post hoc notebooks 06b / 06c (must match exactly) ===
  [PASS       ] 06b evaluable devices                  expected=1689             computed=1689
  [PASS       ] 06b persistent                         expected=893              computed=893
  [PASS       ] 06b % persistent, all                  expected=52.9             computed=52.9
  [PASS       ] 06b % persistent, >=5 y follow-up      expected=58.6             computed=58.6
  [PASS       ] 06b % persistent, prior use            expected=56.4             computed=56.4
  [PASS       ] 06b % persistent, no prior use         expected=47.7             computed=47.7
  [PASS       ] 06c category A citations               expected=2553             computed=2553
  [PASS       ] 06c category B citations               expected=119              computed=119
  [PASS       ] 06c category C citations               expected=159              computed=159
  [PASS       ] 06c category U citations               expected=6                computed

### Tier 4 — design-related recall harm analysis (06e, registered addendum v1.0)
Re-derives the 06e exposure groups with separate code (vectorised merges over the edge list instead
of 06e's per-device loop), recounts outcomes, refits the primary point estimate from the saved device
table, and recomputes the validation measures from the final codes. Compares everything to
`data/design_harm_summary.json`. Run 06e before this cell.

In [7]:
print("\n=== design harm analysis 06e (must match exactly) ===")
if not os.path.exists(os.path.join(DATA_DIR, "design_harm_summary.json")):
    check("design_harm_summary.json present (run 06e first)", True, False)
else:
    import sys
    from statsmodels.duration.hazard_regression import PHReg
    sys.path.insert(0, "design_arm"); from cause_map import DESIGN
    HS = json.load(open(os.path.join(DATA_DIR, "design_harm_summary.json")))
    HD = pd.read_csv(os.path.join(DATA_DIR, "design_harm_devices.csv"), dtype={"k": str, "code": str})

    # exposure groups, re-derived
    raw5 = pd.DataFrame(json.load(gzip.open("snapshot/510k_raw.json.gz", "rt")))[["k_number", "date_received"]]
    rec5 = raw5.drop_duplicates("k_number").assign(rec=lambda d: pd.to_datetime(d.date_received, errors="coerce"))
    rec5 = rec5.set_index("k_number").rec
    pubd = pd.read_csv(os.path.join(DATA_DIR, "recall_public_dates.csv"), dtype=str)
    pubd = pubd.assign(pub=pd.to_datetime(pubd.recall_public_date)).set_index("k_number").pub
    ev5 = pd.read_csv("design_arm/data/recall_events.csv", dtype={"res_event_number": str})
    ev5["init"] = pd.to_datetime(ev5.event_date_initiated, format="%Y-%m-%d", errors="coerce")
    ev5 = ev5[ev5.init.notna() & (ev5.init.dt.year >= 1976)]
    lk5 = pd.read_csv("design_arm/data/event_device_links.csv", dtype=str).merge(
        ev5[["res_event_number", "init", "fda_category"]], on="res_event_number")
    first_design = lk5[lk5.fda_category == DESIGN].groupby("k_number").init.min()

    e5 = edges[["predicate_knumber", "device_knumber"]].drop_duplicates()
    e5 = e5[e5.device_knumber.isin(set(HD.k))].copy()
    e5["rec"] = e5.device_knumber.map(rec5); e5["pub"] = e5.predicate_knumber.map(pubd)
    e5["fd"] = e5.predicate_knumber.map(first_design)
    e5["q"] = e5.pub < e5.rec
    e5["qd"] = e5.q & (e5.fd < e5.rec)
    direct_q = set(e5.loc[e5.q, "device_knumber"]); direct_d = set(e5.loc[e5.qd, "device_knumber"])
    # ancestors at generations 2 and 3 by repeated merges (minimum generation, excluding the device itself)
    par = edges[["predicate_knumber", "device_knumber"]].drop_duplicates().rename(
        columns={"device_knumber": "child", "predicate_knumber": "anc"})
    g1 = par.rename(columns={"child": "k"}).assign(gen=1)
    g2 = g1.merge(par, left_on="anc", right_on="child", suffixes=("_1", ""))[["k", "anc"]].assign(gen=2)
    g3 = g2.merge(par, left_on="anc", right_on="child", suffixes=("_2", ""))[["k", "anc"]].assign(gen=3)
    A = pd.concat([g1, g2, g3]); A = A[A.k != A.anc].sort_values("gen").drop_duplicates(["k", "anc"])
    A = A[A.k.isin(set(HD.k))]
    A["q"] = A.anc.map(pubd) < A.k.map(rec5)
    distant = set(A.loc[A.q & A.gen.isin([2, 3]), "k"])
    grp = np.select([HD.k.isin(direct_d), HD.k.isin(direct_q), HD.k.isin(distant)],
                    ["design", "nondesign", "distant"], "reference")
    for gname in ["design", "nondesign", "distant", "reference"]:
        check(f"06e group size: {gname}", HS["groups_fda"].get(gname, 0), int((grp == gname).sum()))
    check("06e device-level groups identical", True, bool((grp == HD.grp_fda.values).all()))
    check("06e population", HS["population"], len(HD))
    check("06e own-recall events", HS["events_any"], int(HD.ev_any.sum()))
    for term in ["design", "nondesign", "distant"]:
        m = HD.grp_fda == term
        check(f"06e primary events in {term}", HS["models"]["primary"]["terms"][term]["events"], int(HD.ev_any[m].sum()))

    # primary point estimate refit from the saved device table
    X = pd.DataFrame({t: (HD.grp_fda == t) for t in ["design", "nondesign", "distant"]}).astype(float)
    X["yc"] = (HD.year - 2010) / 10; X["yc2"] = X.yc ** 2; X["lnp"] = np.log(HD.npred); X["firm_prior"] = HD.prior.astype(float)
    pm = PHReg(HD.t_any.values, X.values, status=HD.ev_any.values, strata=HD.code.values, ties="efron").fit(disp=False)
    check("06e primary HR (3 dp)", HS["models"]["primary"]["terms"]["design"]["hr"], round(float(np.exp(pm.params[0])), 3))

    # validation measures from the final codes
    VF = pd.read_csv("design_arm/validation/validation_final_codes.csv")
    yes = VF.final_design.eq("yes"); des = VF.fda_cat.eq("design")
    check("06e PPV numerator", HS["validation"]["ppv_fda_design"][0], int(yes[des].sum()))
    check("06e non-design coded design", HS["validation"]["fda_nondesign_coded_design"][0], int(yes[~des].sum()))
    po = (VF.cm_design == VF.gd_design).mean()
    pe = sum((VF.cm_design == c).mean() * (VF.gd_design == c).mean() for c in ["yes", "no", "unclear"])
    check("06e kappa (design, 3 dp)", HS["validation"]["kappa_design"], round((po - pe) / (1 - pe), 3))


=== design harm analysis 06e (must match exactly) ===
  [PASS       ] 06e group size: design                 expected=438              computed=438
  [PASS       ] 06e group size: nondesign              expected=1530             computed=1530
  [PASS       ] 06e group size: distant                expected=2914             computed=2914
  [PASS       ] 06e group size: reference              expected=4046             computed=4046
  [PASS       ] 06e device-level groups identical      expected=True             computed=True
  [PASS       ] 06e population                         expected=8928             computed=8928
  [PASS       ] 06e own-recall events                  expected=852              computed=852
  [PASS       ] 06e primary events in design           expected=60               computed=60
  [PASS       ] 06e primary events in nondesign        expected=153              computed=153
  [PASS       ] 06e primary events in distant          expected=183              computed=183
 

In [8]:
n_pass, n_tot = sum(results), len(results)
print("=" * 62)
print(f"RESULT: {n_pass}/{n_tot} checks passed")
assert n_pass == n_tot, f"{n_tot - n_pass} check(s) FAILED — see above"

RESULT: 45/45 checks passed
